In [1]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import random
import pandas as pd
from tqdm import tqdm

# Loading datasets
families_df = pd.read_excel("labeled_final_dataset_with_pids.xlsx")
images_df = pd.read_excel(r"C:\Users\Harsh Datt\Data Science\3rd Image Recognition\ML_Training_Dataset_Cleaned.xlsx")
images_df = images_df.fillna("")

matched_images_list = []

# Creating a set to track which images have already been assigned
used_images = set()

# Wrapping the loop with tqdm to show a progress bar
for index, row in tqdm(families_df.iterrows(), total=len(families_df), desc="Matching Images"):
    fam_id = row.get("hasfamilyid", f"FAM_{index}")
    is_bpl = row.get("BPL_Target") == 1
    region = "Urban" if row.get("r_u") == "U" else "Rural"
    address = f"{row.get('wardvillage', '')}, {row.get('blocktown', '')}, {row.get('district', '')}"
    image_context = random.choice(
        ["Exterior_House_or_Apartment", "Interior_Flat_or_Room"]
    )

    # Base dictionary structure
    prompt_data = {
        "Family_ID": fam_id,
        "Region": region,
        "Address": address,
        "Image_Context": image_context,
        "Housing_Category": "",
        "Exterior_Features": {},
        "Interior_Features": {},
        "Visible_Assets": None,
        "Structural_Score": 0.0,
        "Structural_Condition": "",
    }

    # CONDITION: BELOW POVERTY LINE (BPL)
    if is_bpl:
        prompt_data["Housing_Category"] = random.choice(
            ["Kutcha", "Semi-Pucca", "Pucca", "Premium"]
        )

        # 1. Structural Condition & Score Matrix Mapping
        struct_cond = random.choice(["Dilapidated", "Poor", "Average", "Excellent"])
        prompt_data["Structural_Condition"] = struct_cond

        if struct_cond == "Excellent":
            prompt_data["Structural_Score"] = round(random.uniform(0.80, 1.00), 2)
            wall_ext, roof_ext = "Finished Concrete/Plaster", "Concrete"
            floor_int, wall_int = "Tiles/Marble", random.choice(["Painted", "Tiles"])
        elif struct_cond == "Average":
            prompt_data["Structural_Score"] = round(random.uniform(0.60, 0.79), 2)
            wall_ext = "Finished Concrete/Plaster"
            roof_ext = random.choice(["Concrete", "Asbestos", "Khaprail"])
            floor_int = random.choice(["Tiles/Marble", "Cement/Concrete"])
            wall_int = random.choice(["Painted", "Raw_Plaster"])
        elif struct_cond == "Poor":
            prompt_data["Structural_Score"] = round(random.uniform(0.40, 0.59), 2)
            wall_ext = random.choice(["Finished Concrete/Plaster", "Exposed Brick"])
            roof_ext = random.choice(
                ["Concrete", "Asbestos", "Khaprail", "Corrugated Tin/Metal"]
            )
            floor_int = random.choice(["Mud/Earth", "Cement/Concrete"])
            # Interior validation logic
            if floor_int == "Mud/Earth":
                wall_int = random.choice(["Painted", "Raw_Plaster", "Bare_Brick"])
            else:
                wall_int = random.choice(["Painted", "Raw_Plaster", "Bare_Brick"])
        else:  # Dilapidated
            prompt_data["Structural_Score"] = round(random.uniform(0.10, 0.39), 2)
            wall_ext = random.choice(["Mud/Makeshift", "Exposed Brick"])
            roof_ext = random.choice(
                ["Thatch/Tarpaulin", "Asbestos", "Khaprail", "Corrugated Tin/Metal"]
            )
            floor_int = "Mud/Earth"
            wall_int = random.choice(["Mud", "Bare_Brick"])

        # 2. Stories Assignment with dynamic probability checks
        rand_val = random.random()
        if region == "Rural":
            stories = "One Storied" if rand_val < 0.60 else random.choice([f"{i} Storied" for i in range(2, 5)])
        else:  # Urban
            stories = "One Storied" if rand_val < 0.30 else random.choice([f"{i} Storied" for i in range(2, 11)])

        # Safety Fallback: Enforcing structural rules over condition rules if conflicts happen
        if stories != "One Storied":
            if roof_ext == "Thatch/Tarpaulin":
                roof_ext = random.choice(["Corrugated Tin/Metal", "Khaprail", "Asbestos", "Concrete"])
            if wall_ext == "Mud/Makeshift":
                wall_ext = random.choice(["Exposed Brick", "Finished Concrete/Plaster"])

        # 3. Context Specific Assembly
        if image_context == "Exterior_House_or_Apartment":
            prompt_data["Exterior_Features"] = {
                "Stories": stories,
                "Roof_Type": roof_ext,
                "Wall_Type": wall_ext,
            }
        else:
            prompt_data["Interior_Features"] = {
                "Floor_Material": floor_int,
                "Wall_Finish": wall_int,
            }

        # 4. Regional Assets Filters & Mud/Brick Restriction
        if region == "Urban":
            asset_pool = ["Two-Wheeler", "Four-Wheeler", "AC"]
        else:
            asset_pool = ["Two-Wheeler", "Four-Wheeler", "Tractor", "Truck"]

        # If mud or bare brick is present anywhere, strip AC options completely
        has_mud_or_brick = any(
            x in [wall_ext, roof_ext, floor_int, wall_int]
            for x in ["Mud/Makeshift", "Thatch/Tarpaulin", "Mud/Earth", "Mud", "Exposed Brick", "Bare_Brick"]
        )
        if has_mud_or_brick and "AC" in asset_pool:
            asset_pool.remove("AC")

        prompt_data["Visible_Assets"] = random.choice(asset_pool)

    # CONDITION: ABOVE POVERTY LINE (NON-BPL)
    else:
        income_range = str(row.get("familyRange", ""))
        has_car = row.get("isFourWheeler", 0) > 0
        has_high_bill = row.get("electric_annualbillamount", 0) > 20000

        is_premium = (
            has_car or "800001" in income_range or "1000000" in income_range
        )
        prompt_data["Housing_Category"] = "Premium" if is_premium else "Pucca"

        # Determining Structural parameters safely
        struct_cond = random.choice(["Average", "Excellent"])
        prompt_data["Structural_Condition"] = struct_cond

        # Rigidly matching engineering definition to Non-BPL features
        if struct_cond == "Excellent":
            prompt_data["Structural_Score"] = round(random.uniform(0.80, 1.00), 2)
            wall_ext, roof_ext = "Finished Concrete/Plaster", "Concrete"
            floor_int, wall_int = "Tiles/Marble", "Tiles" if is_premium else "Painted"
        else:  # Average
            prompt_data["Structural_Score"] = round(random.uniform(0.60, 0.79), 2)
            wall_ext = "Finished Concrete/Plaster"
            roof_ext = random.choice(["Concrete", "Asbestos", "Khaprail"])
            floor_int = "Tiles/Marble" if is_premium else "Cement/Concrete"
            wall_int = "Painted" if is_premium else "Raw_Plaster"

        # Stories Assignment Matrix
        rand_val = random.random()
        if region == "Rural":
            stories = "One Storied" if rand_val < 0.60 else random.choice([f"{i} Storied" for i in range(2, 5)])
        else:
            stories = "One Storied" if rand_val < 0.30 else random.choice([f"{i} Storied" for i in range(2, 11)])

        if image_context == "Exterior_House_or_Apartment":
            prompt_data["Exterior_Features"] = {
                "Stories": stories,
                "Roof_Type": roof_ext,
                "Wall_Type": wall_ext,
            }
        else:
            prompt_data["Interior_Features"] = {
                "Floor_Material": floor_int,
                "Wall_Finish": wall_int,
            }

        # Non-BPL Hardcoded Asset List Strategy
        assets = []
        if has_car:
            assets.append("Four-Wheeler")
        if has_high_bill:
            assets.append("AC")
        prompt_data["Visible_Assets"] = (
            assets if assets else random.choice(["Two-Wheeler", "None"])
        )

    # Image Matching Engine
    context_images = images_df[images_df["Image_Context"] == image_context]
    best_match_name = "No Match Found"
    best_score = -1

    for img_idx, img_row in context_images.iterrows():
        current_img_name = img_row.get("Image_Name", "Unknown")
        
        # Skipping the image if it has already been used
        if current_img_name in used_images:
            continue

        score = 0

        # Structural properties match check
        if img_row.get("Housing_Category") == prompt_data["Housing_Category"]:
            score += 10
        if img_row.get("Ext_Structural_Condition") == prompt_data["Structural_Condition"]:
            score += 5

        # Image view based calculations
        if image_context == "Exterior_House_or_Apartment" and prompt_data["Exterior_Features"]:
            if img_row.get("Ext_Stories") == prompt_data["Exterior_Features"]["Stories"]:
                score += 3
            if img_row.get("Ext_Roof_Type") == prompt_data["Exterior_Features"]["Roof_Type"]:
                score += 3
            if img_row.get("Ext_Wall_Type") == prompt_data["Exterior_Features"]["Wall_Type"]:
                score += 3
        elif image_context == "Interior_Flat_or_Room" and prompt_data["Interior_Features"]:
            if img_row.get("Int_Floor_Material") == prompt_data["Interior_Features"]["Floor_Material"]:
                score += 3
            if img_row.get("Int_Wall_Finish") == prompt_data["Interior_Features"]["Wall_Finish"]:
                score += 3

        # Dynamic asset evaluation checks
        img_assets = str(img_row.get("Visible_Assets_For_Income", ""))
        target_asset = prompt_data["Visible_Assets"]
        if isinstance(target_asset, list):
            for asset in target_asset:
                if asset in img_assets:
                    score += 8
        elif target_asset and target_asset in img_assets:
            score += 8

        if score > best_score:
            best_score = score
            best_match_name = current_img_name

    # Marking the chosen image as used (so no other family can take it)
    if best_match_name != "No Match Found":
        used_images.add(best_match_name)

    matched_images_list.append(best_match_name)

# OUTPUT SCRIPT 
families_df["Matched_Image_Name"] = matched_images_list

final_df = pd.merge(
    families_df,
    images_df,
    left_on="Matched_Image_Name",
    right_on="Image_Name",
    how="left"
)

final_df.to_excel("families_mapped_to_images.xlsx", index=False)
print("Processing finalized successfully!")

Matching Images: 100%|█████████████████████████████████████████████████████████████| 2301/2301 [10:28<00:00,  3.66it/s]


Processing finalized successfully!


In [3]:
import os
import shutil
import pandas as pd
from tqdm import tqdm

# Configuration
source_folder = r"C:\Users\Harsh Datt\Data Science\3rd Image Recognition\IMAGES"
destination_folder = "./Family Images/"
excel_file = "families_mapped_to_images.xlsx"

# Creating the destination folder if it doesn't already exist
os.makedirs(destination_folder, exist_ok=True)

# Processing the Excel File
print("Loading Excel file...")
df = pd.read_excel(excel_file)

# Extracting unique image names
image_names = df['Matched_Image_Name'].dropna().unique()

# Removing "No Match Found" from the list so the script doesn't try to copy it as a file
image_names = [name for name in image_names if name != "No Match Found"]

print(f"Found {len(image_names)} unique images to copy.\n")

# Copying the Images
success_count = 0
missing_images = []

# Wrapping the loop in tqdm for the progress bar
for img_name in tqdm(image_names, desc="Copying Images"):
    
    # Construct the full file paths
    # Note: If your Excel file does NOT include file extensions (like .jpg or .png),
    # you will need to add it here, e.g., str(img_name) + ".jpg"
    file_name = str(img_name)
    
    src_path = os.path.join(source_folder, file_name)
    dest_path = os.path.join(destination_folder, file_name)
    
    # Checking if the file exists in the source folder before copying
    if os.path.exists(src_path):
        shutil.copy2(src_path, dest_path) # copy2 preserves metadata like creation date
        success_count += 1
    else:
        missing_images.append(file_name)

# Summary Report
print("\n--- Copy Summary ---")
print(f"Successfully copied: {success_count} images.")

if missing_images:
    print(f"Warning: Could not find {len(missing_images)} images in the source folder.")
    print(f"Here are a few missing examples: {missing_images[:5]}")

Loading Excel file...
Found 2301 unique images to copy.



Copying Images: 100%|█████████████████████████████████████████████████████████████| 2301/2301 [00:10<00:00, 212.45it/s]


--- Copy Summary ---
Successfully copied: 2301 images.


In [4]:
import os
import pandas as pd

# Configuration
image_folder = "./Family Images/"
excel_file = "families_mapped_to_images.xlsx"
image_column_name = "Matched_Image_Name" # The column containing the image names

# Extracting Image Names from Excel
print("Loading Excel file...")
df = pd.read_excel(excel_file)

# Extracting unique image names and removing 'No Match Found' or empty values
excel_images = set(df[image_column_name].dropna().unique())
if "No Match Found" in excel_images:
    excel_images.remove("No Match Found")
if "Unknown" in excel_images:
    excel_images.remove("Unknown")

# Making sure all items are strings
excel_images = {str(name) for name in excel_images}

# Extracting File Names from the Folder
print(f"Scanning folder: {image_folder}...")
# Gettng a list of all files in the folder. 
# Note: If the excel file does not have file extensions (e.g. .jpg), we might need to strip extensions from the folder files for a fair comparison.
# This assumes the excel file has the exact names as the files in the folder.
if os.path.exists(image_folder):
    folder_images = set(os.listdir(image_folder))
else:
    print(f"Error: Folder '{image_folder}' does not exist.")
    folder_images = set()

# Comparing the Sets
# Images in both Excel and Folder
matched_images = excel_images.intersection(folder_images)

# Images in Excel but MISSING in the Folder
missing_in_folder = excel_images - folder_images

# Images in Folder but NOT in Excel
extra_in_folder = folder_images - excel_images

# Output the Results
print("\n--- Comparison Summary ---")
print(f"Total unique images expected by Excel: {len(excel_images)}")
print(f"Total files found in the folder: {len(folder_images)}")
print(f"Number of successfully matched images: {len(matched_images)}")

print(f"\nImages missing from the folder: {len(missing_in_folder)}")
if missing_in_folder:
    print(f"Examples of missing images: {list(missing_in_folder)[:10]}")

print(f"\nExtra files in the folder (not in Excel): {len(extra_in_folder)}")
if extra_in_folder:
    print(f"Examples of extra files: {list(extra_in_folder)[:10]}")

Loading Excel file...
Scanning folder: ./Family Images/...

--- Comparison Summary ---
Total unique images expected by Excel: 2301
Total files found in the folder: 2301
Number of successfully matched images: 2301

Images missing from the folder: 0

Extra files in the folder (not in Excel): 0
